# Notebook 05 — IoT Pilot Site Prioritisation
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Implement the 100-point scoring framework to rank boreholes for IoT pilot selection.

**Framework source:** Kitui County Borehole Dashboard — Sheet 5 (Ranking Framework)  
**This notebook does not create a new scoring system.** It automates the one Washlab has already built.

---
## Scoring summary
| Points | Criteria | Data status |
|--------|----------|-------------|
| 45 | Population Served (20) + Yield (15) + Ownership (10) | **Available now** |
| 55 | Additionality (20) + Energy (20) + WQ (10) + Rehab Cost (10) + Drinking Water (5) + IoT Viability (5) | **Field assessment required** |

## Priority thresholds
| Threshold | Score |
|-----------|-------|
| First Priority | 80+ / 100 |
| Second Priority Shortlist | 65–79 / 100 |
| Conditional | Below 65 |

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install pandas geopandas matplotlib plotly openpyxl -q

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
from shapely.geometry import Point
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
print('Setup complete')

In [ ]:
# ── 1. Load master dataset ────────────────────────────────────────────────────
# Sheet A for full dataset including county scoring fields
df = pd.read_excel(
    DRIVE + 'Kitui_Boreholes_Master_Dataset.xlsx',
    sheet_name='A_Master_Dataset',
    header=2
)

# Scoring applies to county 34-ward boreholes only
# 6 extra wards (mwater_only) lack county scoring data
county_bh = df[df['Match_Method'].isin(['exact', 'fuzzy', 'no_match'])].copy()

print(f'Total boreholes: {len(df)}')
print(f'County 34-ward (scoreable): {len(county_bh)}')
print(f'6 extra wards (GPS only, county data pending): {len(df) - len(county_bh)}')

In [ ]:
# ── 2. Scoring parameters (from Dashboard Parameters sheet) ──────────────────
# TODO: Load directly from Parameters sheet for single source of truth
# params_df = pd.read_excel(DRIVE + '24_04_2026_Borehole_Dashboard_Summary_40Wards.xlsx',
#                           sheet_name='Parameters', header=2)

# Hardcoded from Parameters sheet — update if dashboard thresholds change
PARAMS = {
    # Population tiers (households)
    'Pop_Tier1_Min': 500,  'Pop_Tier1_Score': 20,
    'Pop_Tier2_Min': 300,  'Pop_Tier2_Score': 15,
    'Pop_Tier3_Min': 150,  'Pop_Tier3_Score': 10,
    'Pop_Tier4_Min': 75,   'Pop_Tier4_Score':  5,
    'Pop_Tier5_Min': 1,    'Pop_Tier5_Score':  2,
    'Pop_Zero_Score': 0,
    # Yield tiers (m³/hr)
    'Yield_Tier1_Min': 5.0,  'Yield_Tier1_Score': 15,
    'Yield_Tier2_Min': 2.5,  'Yield_Tier2_Score': 10,
    'Yield_Tier3_Min': 1.0,  'Yield_Tier3_Score':  5,
    'Yield_Zero_Score': 0,
}
print('Scoring parameters loaded')
print('NOTE: Verify these thresholds against the Parameters sheet before final scoring')

In [ ]:
# ── 3. Scoring functions ──────────────────────────────────────────────────────

def score_population(hhs, p=PARAMS):
    """Criterion 1: Population Served — 20 pts"""
    if pd.isna(hhs) or hhs == 0:
        return p['Pop_Zero_Score']
    elif hhs >= p['Pop_Tier1_Min']: return p['Pop_Tier1_Score']
    elif hhs >= p['Pop_Tier2_Min']: return p['Pop_Tier2_Score']
    elif hhs >= p['Pop_Tier3_Min']: return p['Pop_Tier3_Score']
    elif hhs >= p['Pop_Tier4_Min']: return p['Pop_Tier4_Score']
    elif hhs >= p['Pop_Tier5_Min']: return p['Pop_Tier5_Score']
    return p['Pop_Zero_Score']

def score_yield(yld, p=PARAMS):
    """Criterion 2: Reliable Yield — 15 pts"""
    if pd.isna(yld) or yld == 0:
        return p['Yield_Zero_Score']
    elif yld >= p['Yield_Tier1_Min']: return p['Yield_Tier1_Score']
    elif yld >= p['Yield_Tier2_Min']: return p['Yield_Tier2_Score']
    elif yld >= p['Yield_Tier3_Min']: return p['Yield_Tier3_Score']
    return p['Yield_Zero_Score']

def score_ownership(mgmt):
    """Criterion 3: Borehole Ownership — 10 pts"""
    mapping = {
        'Kitwasco': 10, 'Kimwasco': 10,
        'Project Maji': 8, 'FundiFix': 8,
        'Proposed - Professionalisation': 7,
        'Community': 5, 'Community / School': 5,
        'School / Institution': 3,
        'Church / Faith-Based': 3,
        'Private': 1,
    }
    return mapping.get(str(mgmt), 0)

def score_borehole(row, field_data=None):
    """
    Score a borehole on the full 100-point framework.
    field_data: dict with keys additionality, water_quality, rehab_cost,
                drinking_water, iot_viability, energy_compliance
    Returns dict with total_score, breakdown, and completion status.
    """
    scores = {
        'population':  score_population(row.get('Population_Served_HHs')),
        'yield':       score_yield(row.get('Yield_m3_hr')),
        'ownership':   score_ownership(row.get('Management_Type')),
    }
    field_keys = {
        'additionality': 20, 'water_quality': 10, 'rehab_cost': 10,
        'drinking_water': 5, 'iot_viability': 5, 'energy_compliance': 20
    }
    fd = field_data or {}
    for key in field_keys:
        scores[key] = fd.get(key, None)

    filled   = {k: v for k, v in scores.items() if v is not None}
    total    = sum(filled.values())
    max_pts  = sum(field_keys[k] for k in field_keys if scores[k] is not None) + 45
    pending  = [k for k in field_keys if scores[k] is None]
    max_possible = total + sum(field_keys[k] for k in pending)

    return {
        'total_score':        total,
        'max_available':      max_pts,
        'max_possible':       max_possible,
        'field_pts_pending':  sum(field_keys[k] for k in pending),
        'field_complete':     len(pending) == 0,
        'priority_tier': (
            'First Priority'    if total >= 80 else
            'Second Priority'   if total >= 65 else
            'Conditional'
        ) if len(pending) == 0 else 'Incomplete — field data pending',
        'breakdown': scores,
        'pending_criteria': pending,
    }

print('Scoring functions ready')

In [ ]:
# ── 4. Score all county boreholes on available 45 points ─────────────────────
results = county_bh.apply(lambda row: score_borehole(row.to_dict()), axis=1)

county_bh['Score_45']             = results.apply(lambda x: x['total_score'])
county_bh['Score_Pop']            = results.apply(lambda x: x['breakdown']['population'])
county_bh['Score_Yield']          = results.apply(lambda x: x['breakdown']['yield'])
county_bh['Score_Ownership']      = results.apply(lambda x: x['breakdown']['ownership'])
county_bh['Field_Pts_Pending']    = results.apply(lambda x: x['field_pts_pending'])
county_bh['Priority_Tier']        = results.apply(lambda x: x['priority_tier'])

# Sort by score descending
ranked = county_bh.sort_values('Score_45', ascending=False).reset_index(drop=True)
ranked['Interim_Rank'] = ranked.index + 1

print(f'Scored {len(ranked)} boreholes')
print(f'\nScore distribution (45 pts max):')
print(ranked['Score_45'].describe().round(1))
print(f'\nAt maximum 45/45: {(ranked["Score_45"]==45).sum()}')
print(f'Score >= 35:       {(ranked["Score_45"]>=35).sum()}')
print(f'Score >= 30:       {(ranked["Score_45"]>=30).sum()}')

In [ ]:
# ── 5. View interim top 20 ────────────────────────────────────────────────────
cols = ['Interim_Rank','Ward','Sub_County','Borehole_Name',
        'Score_45','Score_Pop','Score_Yield','Score_Ownership',
        'Field_Pts_Pending','GPS_Quality','Functionality_Status']

print('TOP 20 — INTERIM RANKING (45/100 pts only — field assessment pending)')
print('=' * 80)
display(ranked[cols].head(20))

In [ ]:
# ── 6. Field data input template ─────────────────────────────────────────────
# When field assessment data is available, add it here as a dict:
# Key = Borehole_ID, Value = dict of field scores

# Example (replace with real field data):
FIELD_DATA = {
    # 'KTI-0001': {
    #     'additionality': 15,
    #     'water_quality': 10,
    #     'rehab_cost': None,       # None = not yet assessed
    #     'drinking_water': 5,
    #     'iot_viability': 5,
    #     'energy_compliance': 20,  # Solar
    # },
}

print(f'Field data entries loaded: {len(FIELD_DATA)}')
print('Add field assessment scores to FIELD_DATA dict to update rankings.')
print('Re-run cells 5 and 6 — no other changes needed.')

In [ ]:
# ── 7. Export interim ranked list ─────────────────────────────────────────────
export_cols = ['Interim_Rank','Borehole_ID','Sub_County','Ward','Borehole_Name',
               'Score_45','Score_Pop','Score_Yield','Score_Ownership',
               'Field_Pts_Pending','Priority_Tier',
               'Latitude','Longitude','GPS_Quality',
               'Functionality_Status','Management_Type',
               'Population_Served_HHs','Yield_m3_hr','mWater_ID']

out_path = DRIVE + 'outputs/kitui_borehole_interim_rankings.xlsx'
ranked[export_cols].to_excel(out_path, index=False)
print(f'Exported to: {out_path}')
print('NOTE: Rankings are INTERIM — 55 field-assessed points still pending.')
print('Do not share as final rankings until field assessment is complete.')